# 01 — Exploratory Data Analysis

HousePrice-AI · [Kaggle House Prices — Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques) (Ames Housing dataset).

**What this notebook covers:**
1. First look: shapes, dtypes, summary statistics
2. Missing-value analysis — and why most NaNs are categories, not gaps
3. The target `SalePrice`: distribution, skew, and why we model `log(SalePrice)`
4. Categorical vs numeric variables
5. Correlations with `SalePrice`
6. Feature engineering preview (the output of `src/features/`)

> All numbers below are computed live from `data/raw/`. Figures are saved to `reports/figures/`.

## 0. Setup

```bash
pip install -r requirements.txt
python -m src.pipeline   # optional - the notebook re-runs it inline
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import PATHS, TARGET
from src.data.load_data import load_raw_data

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
PATHS.figures_dir.mkdir(parents=True, exist_ok=True)

print("pandas", pd.__version__, "| numpy", np.__version__)

## 1. First look

In [ ]:
train_raw, test_raw = load_raw_data()
print("train:", train_raw.shape, "| test:", test_raw.shape)
display(train_raw.head())

In [ ]:
train_raw.info()

In [ ]:
display(train_raw.describe().T)

## 2. Missing values — two very different kinds

Per the [data dictionary](https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data), most NaNs are **not** missing data:

- `Alley`, `PoolQC`, `Fence`, `MiscFeature`, `FireplaceQu`, `GarageType`, `GarageFinish`, `GarageQual`, `GarageCond`, `BsmtQual`, `BsmtCond`, `BsmtExposure`, `BsmtFinType1`, `BsmtFinType2` — `NA` means *"this feature does not exist"* (no alley, no basement, no garage...).
- `LotFrontage` is the one column with genuine missingness (259 rows) — imputed with the median of the same `Neighborhood`.
- `Electrical`, `MSZoning`, `Utilities`, ... have a handful of missing rows — filled with the mode.

In [ ]:
miss_train = train_raw.isna().sum()
miss_train = miss_train[miss_train > 0].sort_values(ascending=False)
miss_test = test_raw.isna().sum()
miss_test = miss_test[miss_test > 0]

summary = pd.DataFrame({"train": miss_train, "test": miss_test.reindex(miss_train.index)})
summary = summary.fillna(0).astype(int)
summary["% of train"] = (summary["train"] / len(train_raw) * 100).round(1)
display(summary)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
miss_train.head(20).plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Missing values in train (top 20)")
ax.set_ylabel("rows")
fig.tight_layout()
fig.savefig(PATHS.figures_dir / "missing_values.png", bbox_inches="tight")
plt.show()

In [ ]:
# Prove the 'NA == feature absent' claim directly from the data:
# for every row where the quality column is NaN, the area must be 0.
def na_means_absent(qual_col: str, area_col: str) -> bool:
    mask = train_raw[qual_col].isna()
    return bool((train_raw.loc[mask, area_col] == 0).all())

checks = {
    "PoolQC NaN -> PoolArea == 0": na_means_absent("PoolQC", "PoolArea"),
    "BsmtQual NaN -> TotalBsmtSF == 0": na_means_absent("BsmtQual", "TotalBsmtSF"),
    "GarageType NaN -> GarageArea == 0": na_means_absent("GarageType", "GarageArea"),
}
for label, ok in checks.items():
    print(f"{label}: {ok}")


## 3. The target: `SalePrice`

`SalePrice` is right-skewed (long tail of expensive houses). Kaggle scores **RMSE on `log(SalePrice)`**, so modelling the log is both statistically nicer (skew ~ 0) and aligned with the metric.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(train_raw[TARGET], bins=50, kde=True, ax=axes[0])
axes[0].set_title(f"SalePrice — skew {train_raw[TARGET].skew():.2f}")
sns.histplot(np.log1p(train_raw[TARGET]), bins=50, kde=True, ax=axes[1])
axes[1].set_title(f"log(SalePrice) — skew {np.log1p(train_raw[TARGET]).skew():.2f}")
fig.tight_layout()
fig.savefig(PATHS.figures_dir / "target_distribution.png", bbox_inches="tight")
plt.show()

In [ ]:
display(train_raw[TARGET].describe().round(2).to_frame("SalePrice"))
print("log(SalePrice) mean/std:", round(np.log1p(train_raw[TARGET]).mean(), 4), "/", round(np.log1p(train_raw[TARGET]).std(), 4))

## 4. Categorical vs numeric

`MSSubClass` and `MoSold` are stored as integers but are **nominal codes** (e.g. 60 = "2-story 1946 & newer", 5 = May) — they must not be treated as continuous numbers. Preprocessing casts them to strings.

In [ ]:
obj_cols = train_raw.select_dtypes(include=["object"]).columns
print("object (categorical) columns:", len(obj_cols))
display(train_raw[obj_cols].nunique().sort_values(ascending=False).to_frame("unique values"))

## 5. Correlations with `SalePrice`

In [ ]:
numeric = train_raw.select_dtypes(include=[np.number]).drop(columns=["Id"], errors="ignore")
corr = numeric.corr()[TARGET].sort_values(ascending=False)
display(corr.head(15).to_frame("corr with SalePrice"))

In [ ]:
top_feats = corr.head(11).index.tolist()  # SalePrice + top 10
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(train_raw[top_feats].corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Correlation heatmap — top features")
fig.tight_layout()
fig.savefig(PATHS.figures_dir / "corr_heatmap.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["GrLivArea", "TotalBsmtSF", "OverallQual"]):
    sns.scatterplot(data=train_raw, x=col, y=TARGET, alpha=0.5, ax=ax)
    ax.set_title(f"{col} vs SalePrice")
fig.tight_layout()
fig.savefig(PATHS.figures_dir / "scatter_top.png", bbox_inches="tight")
plt.show()

## 6. Feature engineering preview

`src/features/preprocessing.py` and `src/features/feature_engineering.py` turn raw data into clean, NaN-free features. Engineered columns: `TotalSF`, `TotalBath`, `HouseAge`, `RemodAge`, `GarageAge`, `HasGarage`, `HasPool`, `HasFireplace`, `HasBasement`, `Has2ndFloor`, `TotalPorchSF`, `TotalOutdoorSF`, `QualSF`.

In [ ]:
from src.features.preprocessing import preprocess
from src.features.feature_engineering import engineer_features

train = engineer_features(preprocess(train_raw))
test = engineer_features(preprocess(test_raw))
print("NaNs left: train:", int(train.isna().sum().sum()), "| test:", int(test.isna().sum().sum()))
print("processed shapes:", train.shape, test.shape)

In [ ]:
engineered = [
    "TotalSF", "TotalBath", "HouseAge", "RemodAge", "GarageAge",
    "HasGarage", "HasPool", "HasFireplace", "HasBasement", "Has2ndFloor",
    "TotalPorchSF", "TotalOutdoorSF", "QualSF",
]
display(train[engineered + [TARGET]].corr()[TARGET].sort_values(ascending=False).to_frame("corr with SalePrice"))

## 7. Summary & next steps

- 1460 train / 1459 test rows, 79 raw features; target is right-skewed (`log` transform justified and matches the Kaggle metric).
- Most NaNs are *"feature absent"* categories → handled as categories, not imputed; `LotFrontage` imputed by neighborhood median; a final check guarantees zero NaNs.
- Top raw correlates: `OverallQual` (0.79), `GrLivArea` (0.71), `GarageCars` (0.64), `GarageArea` (0.62), `TotalBsmtSF` (0.61).
- Engineered `QualSF` (quality × total SF) correlates ~0.88 — the best single feature so far.

**Next (Phase 3):** model training with cross-validation on `log(SalePrice)`, model comparison, hyperparameter tuning, and saving the best model. Processed data is persisted via `python -m src.pipeline`.